# Chương 1 bài 2 Thống kê mô tả tiểu đường

**Nguồn:** Lab EDA, mục 1.1.4, trang 4

**Đề bài:** Thực hiện thống kê mô tả trên tập dữ liệu bệnh tiểu đường.

Code triển khai bài tập theo dữ liệu đã cung cấp; không phải đoạn chép nguyên văn bài mẫu.

In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data').is_dir() and (p / 'README.md').is_file())
DATA = ROOT / 'data'
OUT = ROOT / 'ket_qua' / 'C1_BT02_ThongKe_TieuDuong'
OUT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (8, 4)})

def save_fig(name):
    plt.tight_layout()
    plt.savefig(OUT / name, dpi=140, bbox_inches='tight')
    plt.show()

def save_summary(values):
    (OUT / 'tom_tat.json').write_text(
        json.dumps(values, ensure_ascii=False, indent=2), encoding='utf-8'
    )


## 1 Đọc đủ 768 dòng và kiểm tra số 0

In [2]:
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
raw = pd.read_csv(DATA / 'pima-indians-diabetes.csv', header=None, names=columns)
invalid_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_counts = raw[invalid_zero].eq(0).sum()
df = raw.copy()
df[invalid_zero] = df[invalid_zero].replace(0, np.nan)
print('Raw shape:', raw.shape)
print('Zero placeholders:\n', zero_counts)
print('Class distribution:\n', df['Outcome'].value_counts())

Raw shape: (768, 9)
Zero placeholders:
 Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64
Class distribution:
 Outcome
0    500
1    268
Name: count, dtype: int64


## 2 Thống kê sau khi đánh dấu giá trị không hợp lệ

In [3]:
numeric = df.select_dtypes(include='number')
summary = numeric.describe().T
summary['mode_first'] = numeric.mode().iloc[0]
summary['variance_sample'] = numeric.var(ddof=1)
summary['std_sample'] = numeric.std(ddof=1)
summary['variance_population'] = numeric.var(ddof=0)
summary['std_population'] = numeric.std(ddof=0)
summary['range'] = numeric.max() - numeric.min()
summary['percentile_60'] = numeric.quantile(0.60)
summary['IQR'] = numeric.quantile(0.75) - numeric.quantile(0.25)
summary['missing'] = numeric.isna().sum()
q1, q3 = numeric.quantile(0.25), numeric.quantile(0.75)
iqr = q3 - q1
summary['outside_IQR_fences'] = ((numeric < q1 - 1.5 * iqr) |
                                 (numeric > q3 + 1.5 * iqr)).sum()
print(summary.round(4).to_string())
summary.to_csv(OUT / 'thong_ke_mo_ta.csv', encoding='utf-8-sig')

                          count      mean       std     min      25%       50%       75%     max  mode_first  variance_sample  std_sample  variance_population  std_population    range  percentile_60       IQR  missing  outside_IQR_fences
Pregnancies               768.0    3.8451    3.3696   0.000   1.0000    3.0000    6.0000   17.00       1.000          11.3541      3.3696              11.3393          3.3674   17.000         4.0000    5.0000        0                   4
Glucose                   763.0  121.6868   30.5356  44.000  99.0000  117.0000  141.0000  199.00      99.000         932.4254     30.5356             931.2033         30.5156  155.000       125.0000   42.0000        5                   0
BloodPressure             733.0   72.4052   12.3822  24.000  64.0000   72.0000   80.0000  122.00      70.000         153.3178     12.3822             153.1087         12.3737   98.000        75.0000   16.0000       35                  14
SkinThickness             541.0   29.1534   10.4

## 3 So sánh trước và sau xử lý

In [4]:
comparison = pd.DataFrame({'mean_raw': raw[invalid_zero].mean(),
                           'mean_excluding_invalid_zero': df[invalid_zero].mean(),
                           'invalid_zero_count': zero_counts})
print(comparison.round(3))
comparison.to_csv(OUT / 'so_sanh_so_0.csv', encoding='utf-8-sig')
save_summary({'rows': len(df), 'columns': df.shape[1],
              'class_counts': {str(k):int(v) for k,v in df.Outcome.value_counts().items()},
              'invalid_zeros': {k:int(v) for k,v in zero_counts.items()},
              'glucose_mean_raw': float(raw.Glucose.mean()),
              'glucose_mean_clean': float(df.Glucose.mean()),
              'BMI_mean_clean': float(df.BMI.mean())})

               mean_raw  mean_excluding_invalid_zero  invalid_zero_count
Glucose         120.895                      121.687                   5
BloodPressure    69.105                       72.405                  35
SkinThickness    20.536                       29.153                 227
Insulin          79.799                      155.548                 374
BMI              31.993                       32.457                  11


## Nhận xét và giới hạn
Không xem Pregnancies=0 hoặc Outcome=0 là dữ liệu thiếu. Các số 0 không hợp lý ở 5 chỉ số được chuyển NaN; thống kê bỏ qua NaN, chưa điền trung vị để tránh thay đổi phân bố mô tả. Outcome là nhãn nhị phân, không dùng làm đặc trưng khi xây mô hình.